In [1]:
import torch
import numpy as np
import re
import copy
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict, Any, Literal
from transformers import pipeline
import json, os
import warnings
from embeddings import *
import math
from templates import *
from datetime import datetime
warnings.filterwarnings("ignore")

/home/m.gromadzki/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4373.03it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: ../../biobert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED | 

In [2]:
text_vectorstore = FAISS.load_local("../data/vdbs/text_vdb", TEXT_EMBEDDINGS, allow_dangerous_deserialization=True)
image_vectorstore = FAISS.load_local("../data/vdbs/img_vdb", IMAGE_EMBEDDINGS, allow_dangerous_deserialization=True)

In [ ]:
class AgentState(TypedDict):
    subject_id: int
    action: Literal["search_text", "search_imaging", "finish"]
    query: str
    allowed_years: int
    retrieved_docs: List[List[Document]]
    retrieved_docs_str: str
    num_retriev_text: int
    num_retriev_img: int

    # Stage 1
    template: Dict[str, Any]
    action_history: List[Dict[str, Any]]
    step: int

    # Stage 2
    question: str
    chat_history: List[str]
    answer_llm: str

MAX_STEPS = 3
initial_state = {
    "subject_id": 13221453,
    "template": SUMMARY_TEMPLATE,
    "action": None,
    "query": "",
    "allowed_years": 0,
    "retrieved_docs": [],
    "retrieved_docs_str": "",
    "action_history": [],
    "step": 0,
    "num_retriev_text": 3,
    "num_retriev_img": 1,
}

In [4]:
pipe = pipeline(
    "image-text-to-text",
    model="../../medgemma4b",
    dtype=torch.bfloat16,
    device="cuda:3",
    max_new_tokens=32000,
    max_length = None
)

Loading weights:   5%|▍         | 40/883 [00:00<00:00, 1337.02it/s, Materializing param=model.language_model.layers.2.self_attn.v_proj.weight]          

Loading weights: 100%|██████████| 883/883 [00:00<00:00, 3508.20it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      
The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Passing `generation_config` together with generation-related arguments=({'max_length', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
def extract_response_json(text):
    text = text.replace("null", "None")
    try:
        clean = re.sub(r"^```json\s*|\s*```$", "", text.split("<unused95>")[1].strip())
    except:
        clean = re.sub(r"^```json\s*|\s*```$", "", text.strip())
    
    return clean

def node_reason_and_plan(state: AgentState) -> Dict:
    if state["step"] >= MAX_STEPS:
        return {"action": "finish", "query": None, "action_history": state["action_history"] + ["finish"]}

    PROMPT = (
        f"CURRENT CLINICAL SUMMARY:\n{state['template']}\n\n"
        f"PAST ACTIONS:\n{(state['action_history'])}\n\n"
        "You are an autonomous clinical query agent building a patient's pre-visit clinical summary. "
        "Your role is to generate **one focused query** per turn that will be used to retrieve information from the hospital EHR, "
        "which includes structured data and physician notes (e.g., diagnoses, procedures, medications, labs, encounters, discharge summaries).\n\n"

        "Available actions:\n"
        "1. search_text - produce a query for clinical text or structured EHR data;\n"
        "2. search_imaging - produce a query for imaging impressions if clearly needed;\n"
        "3. finish - stop if the summary is clinically sufficient.\n\n"

        "Task: Identify the single most important missing clinical fact for physician decision-making "
        "and produce ONE focused action with a corresponding query. Do not repeat queries from PAST ACTIONS. "
        "Aim for completeness of the templae rather than detail information. \n\n"

        "Rules:\n"
        "- Generate only one action and query per turn.\n"
        "- Do NOT repeat past actions listed above.\n"
        "- Prefer search_text over search_imaging unless imaging is essential.\n"
        "- Do NOT modify the clinical summary.\n"
        "- Do NOT include time expressions inside the query text (e.g., 'recent', 'last year', 'in the past 6 months').\n"
        "- Instead, control temporal scope using the 'allowed_years' parameter.\n"
        "- By default, assume clinical relevance is time-sensitive and include an 'allowed_years' field.\n"
        "- Omit 'allowed_years' ONLY if the query is explicitly historical, foundational, or lifetime in scope "
        "(e.g., initial diagnosis date, past surgical history, genetic conditions).\n\n"

        "Temporal Guidance:\n"
        "- Labs, medications, vitals, imaging, admissions, and active conditions should almost always include 'allowed_years'.\n"
        "- Chronic disease monitoring typically uses 1-3 years.\n"
        "- Medication lists typically use 1-2 years.\n"
        "- Imaging or procedures may use 2-5 years depending on relevance.\n"
        "- Use clinical judgment to choose the smallest reasonable window that answers the question.\n\n"

        "Stop: choose 'finish' if key clinical information is complete, "
        "or prior searches added nothing useful. Do not generate additional actions after choosing finish.\n\n"

        "Output JSON only.\n"
        "Schema: {\"action\": , \"query\": , \"allowed_years\": }\n"
        "- Include 'allowed_years' in most searches.\n"
        "- If omitted, it must be clearly justified by the lifelong or non-time-bounded nature of the query.\n\n"

        "Examples (each example returns only ONE action):\n"
        "1. {\"action\": \"search_text\", \"query\": \"medication list with doses\", \"allowed_years\": 2}\n"
        "2. {\"action\": \"search_text\", \"query\": \"HbA1c results\", \"allowed_years\": 1}\n"
        "3. {\"action\": \"search_imaging\", \"query\": \"echocardiogram impression\", \"allowed_years\": 3}\n"
        "4. {\"action\": \"search_text\", \"query\": \"initial diagnosis date of rheumatoid arthritis\"}\n"
        "5. {\"action\": \"finish\", \"query\": \"\"}"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT}
            ],
        },
    ]

    response = pipe(messages, do_sample=False, max_new_tokens=2_000)
    print(response[0]["generated_text"][-1]["content"])
    clean = extract_response_json(response[0]["generated_text"][-1]["content"])
    plan = eval(clean)
    print("@" * 100)
    print(plan, state["action_history"])
    
    return {
        "action": plan["action"],
        "query": plan["query"],
        "allowed_years": plan.get("allowed_years", None),
        "action_history": state["action_history"] + [plan],
        "step": state["step"] + 1
    }

In [6]:
item2str = {
    "document_id": "Document ID",
    "hadm_id": "Admission ID",
    "admittime": "Admission Time",
    "dischtime": "Discharge Time",
    "section": "Document Type",
    "impression": "Xray Impression",
    "findings": "Xray Findings",
}

def textdoc2str(doc):
    res = "\n".join(f"{item2str[item]}: {doc.metadata[item]}" for item in ["document_id", "hadm_id", "admittime", "dischtime", "section"])
    res += "\nDocument Content:" + doc.page_content
    return res

def imagedoc2str(doc):
    res = "\n".join(f"{item2str[item]}: {doc.metadata[item]}" for item in ["doc_id", "date", "impression", "findings"])
    return res

def windowed_time_decay(doc_date_str, allowed_years, lambda_inside=0.005, lambda_outside=0.03):
    doc_date = datetime.strptime(doc_date_str, "%Y-%m-%d %H:%M:%S")
    now = datetime.now()
    age_days = (now - doc_date).days
    cutoff_days = allowed_years * 365
    if age_days <= cutoff_days:
        return math.exp(-lambda_inside * age_days)
    else:
        return math.exp(-lambda_outside * age_days)
    
def node_text_vector_search(state: AgentState) -> Dict:
    candidates = text_vectorstore.similarity_search_with_relevance_scores(
        state["query"], k=50, filter = {"subject_id": state["subject_id"]}
    )
    results = []
    for doc, sim_score in candidates:
        date_str = doc.metadata.get("admittime", None)

        if date_str and state["allowed_years"]:
            time_weight = windowed_time_decay(date_str, state["allowed_years"])
        else:
            time_weight = 1.0

        # Combine semantic + temporal
        final_score = sim_score * time_weight

        results.append({
            "doc": doc,
            "final_score": final_score
        })

    results.sort(key=lambda x: x["final_score"], reverse=True)
    retrieved_docs_new = [r["doc"] for r in results[:state["num_retriev_text"]]]

    retrieved_docs_str = "\n\n".join(f"Document {i+1}:\n{textdoc2str(doc)}"
        for i, doc in enumerate(retrieved_docs_new)
    )
    retrieved_docs = state["retrieved_docs"]
    retrieved_docs.append(retrieved_docs_new)
    
    return {"retrieved_docs": retrieved_docs, "retrieved_docs_str": retrieved_docs_str}

def node_image_vector_search(state: AgentState) -> Dict:
    retrieved_docs_new = image_vectorstore.similarity_search(state["query"], k=state["num_retriev_img"], filter = {"subject_id": state["subject_id"]})
    retrieved_docs_str = "\n\n".join(imagedoc2str(doc) for doc in retrieved_docs_new)
    
    retrieved_docs = state["retrieved_docs"]
    retrieved_docs.append(retrieved_docs_new)
    
    return {"retrieved_docs": retrieved_docs, "retrieved_docs_str": retrieved_docs_str}

In [7]:
def node_update_template(state: AgentState) -> Dict:
    PROMPT = (
        "SYSTEM INSTRUCTION: think silently if needed.\n\n" 
        f"NEWELY RETRIEVED DOCUMENTS:\n{state["retrieved_docs_str"]}\n\n"
        f"CURRENT CLINICAL SUMMARY\n{state['template']}\n\n"
        "You are a clinical information extraction agent. Update (not rewrite) the existing summary "
        "using ONLY newly retrieved documents.\n\n"
        "Extraction rules: use only information explicitly in the documents, do NOT infer or normalize, "
        "preserve existing fields unless new data adds clarity, include conflicting info with separate evidence.\n\n"
        "Some or all of the provided data in some documents may not be necessary."
        "Evidence rules: every new fact MUST have an evidence entry, return summary unchanged if no new info.\n\n"
        "Formatting rules: return ONLY valid JSON, match summary schema exactly, no extra text, "
        "keys and string values MUST use double quotes. DO NOT EDIT ANY KEYS OF THE SUMMARY."
        "For each key keep a maximum of 5 most significant and newest entries."
        "Empty field should be signified as empty strings or None, "
        "DO NOT use null in any part of the template."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT}
            ],
        },
    ]
    response = pipe(messages, do_sample=False, max_new_tokens=10_000)
    print(response[0]["generated_text"][-1]["content"])
    clean = extract_response_json(response[0]["generated_text"][-1]["content"])
    template = eval(clean)
    return {"template": template}

def route(state: AgentState) -> str:
    return state["action"]

def node_display_summary(state: AgentState) -> None:
    print(state["template"])
    return

In [8]:
graph = StateGraph(AgentState)

graph.add_node("reason_and_plan", node_reason_and_plan)
graph.add_node("text_vector_search", node_text_vector_search)
graph.add_node("image_vector_search", node_image_vector_search)
graph.add_node("update_template", node_update_template)
graph.add_node("display_summary", node_display_summary)

graph.add_edge(START, "reason_and_plan")

graph.add_conditional_edges(
    "reason_and_plan",
    route,
    {
        "search_text": "text_vector_search",
        "search_imaging": "image_vector_search",
        "finish": "display_summary"
    }
)

graph.add_edge("text_vector_search", "update_template")
graph.add_edge("image_vector_search", "update_template")
graph.add_edge("update_template", "reason_and_plan")
graph.add_edge("display_summary", END)

compiled = graph.compile()

In [ ]:
result = compiled.invoke(copy.deepcopy(initial_state))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<unused94>thought
The user wants me to act as an autonomous clinical query agent to build a pre-visit clinical summary.
I need to identify the single most important missing clinical fact for the physician.
Then, I must generate ONE focused action with a corresponding query.
The query should be for structured EHR data or clinical text.
I should not repeat past actions.
I need to choose the smallest reasonable time window for 'allowed_years' based on the clinical context.
I should not include time expressions in the query.
I should not modify the summary.
I should not include 'allowed_years' unless necessary for time-sensitive information.
I should stop if the summary is complete or if previous searches were unhelpful.

Current Clinical Summary:
{'active_problems': [{'problem': '', 'status': '', 'evidence': [{'source_id': '', 'date': ''}]}], 'recent_events': [{'event': '', 'date': '', 'evidence': [{'source_id': '', 'date': ''}]}], 'medications': [{'name': '', 'dose': '', 'route': '', 'ev

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<unused94>thought
The user wants me to update a clinical summary based on three new documents. I need to extract relevant information from each document and update the existing summary fields.

**Plan:**

1.  **Analyze Document 1 (Discharge Summary):**
    *   Extract patient details (ID, Admission/Discharge times).
    *   Extract active problems: Hypertension, Dementia, Metabolic Acidosis, CKD.
    *   Extract medications: Clonidine (patch), Amlodipine, Carvedilol, Aspirin, Plavix.
    *   Extract procedures/events: Placement of J/G-tube, Hydration with IVF containing bicarb.
    *   Extract discharge instructions: Hold Lasix, restart tube feeds when euvolemic.
    *   Extract relevant notes: NPO due to aspiration, improved mental status, dysarthria.

2.  **Analyze Document 2 (Discharge Summary):**
    *   Extract patient details (ID, Admission/Discharge times).
    *   Extract active problems: Dementia, Aspiration (significant), R occipital and R temporal lobe subacute stroke.
    *

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<unused94>thought
The user wants me to identify the single most important missing clinical fact for the physician and generate a focused query to retrieve it.

1.  **Analyze the current summary:** The summary lists active problems (Hypertension, Dementia, Metabolic Acidosis, CKD, Aspiration, Hypothyroidism, GERD), medications, procedures, recent events, and a key lab result (Creatinine).
2.  **Identify missing information:** The summary mentions 'Dementia' as an active problem. However, it doesn't provide any details about the type of dementia, its severity, or any recent changes in its status or management. This is a critical piece of information for managing a patient with dementia, especially considering the aspiration risk.
3.  **Determine the most important missing fact:** Understanding the specific type and severity of dementia is crucial for planning care, including managing aspiration risk and potential cognitive decline.
4.  **Formulate a query:** A query to retrieve the speci

SyntaxError: invalid decimal literal (<string>, line 34)

In [ ]:
class AgentState(TypedDict):
    subject_id: int
    action: Literal["search_text", "search_imaging", "finish"]
    query: str
    allowed_years: int
    retrieved_docs: List[List[Document]]
    retrieved_docs_str: str
    num_retriev_text: int
    num_retriev_img: int

    # Stage 1
    summary: Dict[str, Any]
    action_history: List[Dict[str, Any]]
    step: int

    # Stage 2
    question: str
    chat_history: List[str]
    answer_llm: str

MAX_STEPS = 10
initial_state = {
    "summary": STRUCTURED_TEMPLATE,
    "action": "",
    "query": "",
    "retrieved_docs": [],
    "retrieved_docs_str": "",
    "action_history": [],
    "step": 0,

    "question": "",
    "chat_history": [],
    "answer_llm": "",
}

START_PROMPT = (
    f"PATIENT CLINICAL SUMMARY:\n{STRUCTURED_TEMPLATE}\n\n"
    "Answer the user question based only on the retrieved documents and clinical summary. "
    "Evidence rules: Each claim must be supported by Document ID and date."
)

In [ ]:
def node_get_question(state: AgentState) -> Dict:
    question = input("Question")
    return {"question": question, "num_retriev_text": 10, "num_retriev_img": 3}

def end_chat(state: AgentState) -> str:
    if len(state["question"]) == 0:
        return "end_chat"
    else:
        return "continue_chat"

def node_make_query(state: AgentState) -> Dict:
    PROMPT = (
        f"PATIENT CLINICAL SUMMARY:\n{state['summary']}\n\n"
        f"CHAT HISTORY:\n{state['chat_history']}\n\n"
        f"LATEST USER QUESTION:\n{state['question']}\n\n"

        "You are a clinical retrieval query generator.\n\n"

        "Your task is to analyze the clinical summary, chat history, "
        "and latest user question to form ONE focused, standalone medical search query "
        "optimized for semantic vector retrieval.\n\n"

        "The query must:\n"
        "- Incorporate relevant context from chat history\n"
        "- Be fully self-contained and understandable without conversation context\n"
        "- Focus on one high-priority clinical topic\n"
        "- Use concise medical terminology\n"
        "- NOT include natural-language time expressions inside the query text "
        "(e.g., 'recent', 'last year', 'past 6 months')\n\n"

        "Available actions:\n"
        "1. search_text - query clinical notes or structured EHR data (default/preferred);\n"
        "2. search_imaging - query imaging impressions only if clearly essential.\n\n"

        "Temporal rules:\n"
        "- Control time scope using the 'allowed_years' parameter instead of writing time expressions in the query.\n"
        "- Labs, medications, vitals, imaging, admissions, and active conditions usually require 'allowed_years'.\n"
        "- Chronic disease monitoring: typically 1-3 years.\n"
        "- Medications: typically 1-2 years.\n"
        "- Imaging/procedures: typically 2-5 years depending on relevance.\n"
        "- Use the smallest reasonable time window that answers the clinical question.\n"
        "- Omit 'allowed_years' ONLY if the query is lifelong, foundational, or explicitly historical "
        "(e.g., initial diagnosis date, past surgical history, genetic condition).\n\n"

        "Generate only ONE action and corresponding query.\n"
        "Do NOT modify the clinical summary.\n"
        "Do NOT explain your reasoning.\n\n"

        "Return JSON only using this schema:\n"
        "{\"action\": \"search_text | search_imaging\", "
        "\"query\": \"standalone medical query\", "
        "\"allowed_years\": number (omit only if clearly justified)}"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT}
            ],
        },
    ]

    response = pipe(messages)
    clean = extract_response_json(response[0]["generated_text"][-1]["content"])
    plan = eval(clean)
    print(plan)
    
    return {
        "action": plan["action"],
        "query": plan["query"],
        "allowed_years": plan.get("allowed_years", None),
    }

In [ ]:
def node_route_answer_llm(state: AgentState) -> Dict:
    PROMPT = (
        f"PATIENT CLINICAL SUMMARY:\n{state['summary']}\n\n"
        f"CHAT HISTORY:\n{state['chat_history']}\n\n"
        f"RETRIEVED DOCUMENTS:\n{state['retrieved_docs_str']}\n\n"
        f"LATEST USER QUESTION:\n{state['question']}\n\n"

        "You are an expert LLM router.\n\n"

        "Your task is to analyze the patient clinical summary, chat history, "
        "retrieved documents, and latest user question, "
        "and decide which model should generate the final answer.\n\n"

        "Available models:\n"
        "- medgemma: Use for general medical knowledge, clinical reasoning, "
        "diagnosis, treatment interpretation, and any physician-level medical question.\n"
        "- txgemma: Use for specialized therapeutic and drug discovery related tasks, "
        "including predictive or property analysis (e.g., molecular properties, drug-target interactions), "
        "conversation about therapeutic development contexts, or other research-focused interactions informed by therapeutic data.\n\n"

        "Routing rules:\n"
        "- Prioritize the intent of the latest user question.\n"
        "- If the question requires broad medical knowledge, clinical interpretation, or patient-centered reasoning → choose medgemma.\n"
        "- If the question specifically involves therapeutic discovery, drug properties, biological prediction tasks, "
        "or research-oriented therapeutic dialogue → choose txgemma.\n"
        "- When in doubt and the question is not therapeutic research-focused, prefer medgemma.\n\n"

        "Return JSON only using this schema:\n"
        "{\"answer_llm\": \"medgemma | txgemma\"}\n"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT}
            ],
        },
    ]
    response = pipe(messages)
    clean = extract_response_json(response[0]["generated_text"][-1]["content"])
    plan = eval(clean)

    return {"answer_llm": plan["answer_llm"]}


def node_answer_question_medgemma(state: AgentState) -> Dict:
    PROMPT = (
        f"USER QUESTION: {state["question"]}"
        f"RETRIEVED DOCUMENTS:\n{state["retrieved_docs_str"]}"
    )
    user_prompt = {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT if len(state["chat_history"]) > 0 else START_PROMPT + PROMPT}
            ],
        }
    
    messages = state["chat_history"]
    messages.append(user_prompt)
    response = pipe(messages)
    print(response[0]["generated_text"][-1]["content"])
    
    assistant_reponse = {
            "role": "assistant",
            "content": [
                {"type": "text", "text": response}
            ],
        }
    
    messages.append(assistant_reponse)

    return {"chat_history": messages}

def route_answer_llm(state: AgentState) -> str:
    return state["answer_llm"]

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("get_question", node_get_question)
graph.add_node("make_query", node_make_query)
graph.add_node("text_vector_search", node_text_vector_search)
graph.add_node("image_vector_search", node_image_vector_search)
graph.add_node("route_answer_llm", node_route_answer_llm)
graph.add_node("answer_question_medgemma", node_answer_question_medgemma)
# graph.add_node("answer_question_txgemma", node_answer_question_txgemma)

graph.add_edge(START, "get_question")
graph.add_conditional_edges("get_question",
    end_chat,
    {
        "continue_chat": "make_query",
        "end_chat": END,
    }
)

graph.add_conditional_edges(
    "make_query",
    route,
    {
        "search_text": "text_vector_search",
        "search_imaging": "image_vector_search",
    }
)

graph.add_edge("text_vector_search", "route_answer_llm")
graph.add_edge("image_vector_search", "route_answer_llm")

graph.add_conditional_edges(
    "route_answer_llm",
    route_answer_llm,
    {
        "medgemma": "answer_question_medgemma",
        # "txgemma": "answer_question_txgemma",
    }
)

graph.add_edge("answer_question_medgemma", "get_question")
# graph.add_edge("answer_question_txgemma", "get_question")

compiled = graph.compile()

In [ ]:
result = compiled.invoke(copy.deepcopy(initial_state))

In [ ]:
clinical_record_template = {
  "summary": "",
  "key_points": [],
  "symptoms": [],
  "history": [],
  "medications": [],
  "allergies": [],
  "exam_findings": [],
  "assessment": [],
  "plan": [],
  "open_questions": []
}

In [ ]:
import librosa

In [ ]:
model_id = "../medasr"
asr = pipeline("automatic-speech-recognition", model=model_id)

In [ ]:
audio, SAMPLE_RATE = librosa.load("test_audio.wav", sr=16000)

In [ ]:
SAMPLE_RATE = 16000
CHUNK_SECONDS = 10
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS
OVERLAP_RATIO = 0.1  # 10% overlap
STEP_SAMPLES = int(CHUNK_SAMPLES * (1 - OVERLAP_RATIO))

In [ ]:
class GraphState(TypedDict):
    audio_chunk: np.ndarray
    transcript_chunk: str
    full_transcript: List[str]
    structured_json: str


def node_transcribe(state: GraphState):
    waveform = state["audio_chunk"]

    result = asr(waveform, sampling_rate=SAMPLE_RATE)
    text = result["text"]

    updated_transcript = state["full_transcript"] + [text]

    return {
        "transcript_chunk": text,
        "full_transcript": updated_transcript
    }


def node_summarize(state: GraphState):
    full_text = "\n".join(state["full_transcript"])

    PROMPT = (
        "TRANSCRIPT:\n"
        f"{full_text}\n\n"
        "CONVERSATION SUMMARY:\n"
        f"{state['structured_json']}\n\n"
        "You are a clinical documentation assistant.\n"
        "Update the conversation summary based on the provided transcript.\n\n"
        "RULES:\n"
        "- Add new information.\n"
        "- Update fields if clarified.\n"
        "- Do NOT remove confirmed facts unless explicitly corrected.\n"
        "- If clinician asks unanswered questions, add to \"open_questions\".\n"
        "- Do NOT hallucinate missing values.\n"
        "- Output ONLY valid JSON.\n"
        "- Do NOT include explanations.\n"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT}
            ],
        },
    ]

    response = pipe(messages, max_new_tokens=2000)

    output = extract_response_json(response[0]["generated_text"][-1]["content"])

    print(f"\n[MedGemma Summary]\n{output}\n")

    return {"structured_json": output}

In [ ]:
builder = StateGraph(GraphState)

builder.add_node("transcribe", node_transcribe)
builder.add_node("summarize", node_summarize)

builder.add_edge(START, "transcribe")
builder.add_edge("transcribe", "summarize")
builder.add_edge("summarize", END)

graph = builder.compile()

In [ ]:
structured_json_memory = copy.deepcopy(clinical_record_template)
transcript_memory = []

total_samples = len(audio)
start = 0

while start < total_samples:
    end = min(start + CHUNK_SAMPLES, total_samples)
    chunk = audio[start:end]

    result = graph.invoke({
        "structured_json": structured_json_memory,
        "audio_chunk": chunk,
        "full_transcript": transcript_memory
    })

    structured_json_memory = result["structured_json"]
    transcript_memory = result["full_transcript"]

    start += STEP_SAMPLES  # move forward with overlap

In [ ]:
transcript_memory

In [ ]:
"".join(item[:-4] for item in transcript_memory)

In [ ]:
structured_json_memory